# Workshop – Nonlinear features, regularization and hyperparameter search

In this workshop we'll build a series of linear models that predict how fast diabetes progresses, and use them to explore three ideas:

1. **Nonlinear features**: adding squares and products of the predictors to a linear model;
2. **Regularization**: keeping the parameters small to avoid overfitting (Ridge and Lasso regression);
3. **Hyperparameter search**: choosing settings such as the polynomial degree and the amount of regularization in a principled way.

### The data

442 patients with diabetes were measured at the start of a study: age, sex, body mass index (BMI), average blood pressure, and six blood serum measurements.  One year later, a doctor scored how much the disease had **progressed**: a number between 25 and 346, where higher means worse.  Can we predict this progression from the baseline measurements?

The data is from eEfron, Hastie, Johnstone and Tibshirani, *Least Angle Regression*, Annals of Statistics, 2004.  This is the dataset with which the authors illustrated the Lasso, a form of linear regression with variable selection - see below.

## 0. Loading the data

The dataset comes with scikit-learn, so no download is needed.  The original column names `s1` … `s6` are not very informative, so we rename them.  The sex of the patients is coded as 1 and 2 (the documentation does not say which is which); since it is a category and not a quantity, we turn it into text.

★ Run the cell.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

diabetes = load_diabetes(as_frame=True, scaled=False)
X = diabetes.data.rename(columns={"s1": "chol", "s2": "ldl", "s3": "hdl",
                                  "s4": "chol_hdl", "s5": "log_trig", "s6": "glucose"})
X["sex"] = X["sex"].map({1.0: "group 1", 2.0: "group 2"})
y = diabetes.target.rename("progression")
X.head()

,age,sex,bmi,bp,chol,ldl,hdl,chol_hdl,log_trig,glucose
0,59.0,group 2,32.1,101.0,157.0,93.2,38.0,4.0,4.8598,87.0
1,48.0,group 1,21.6,87.0,183.0,103.2,70.0,3.0,3.8918,69.0
2,72.0,group 2,30.5,93.0,156.0,93.6,41.0,4.0,4.6728,85.0
3,24.0,group 1,25.3,84.0,198.0,131.4,40.0,5.0,4.8903,89.0
4,50.0,group 1,23.0,101.0,192.0,125.4,52.0,4.0,4.2905,80.0


| column | meaning |
|---|---|
| `age` | age in years |
| `sex` | sex (group 1 or group 2) |
| `bmi` | body mass index (kg/m²) |
| `bp` | average blood pressure (mm Hg) |
| `chol` | total serum cholesterol |
| `ldl` | LDL cholesterol ("bad" cholesterol) |
| `hdl` | HDL cholesterol ("good" cholesterol) |
| `chol_hdl` | ratio of total cholesterol to HDL |
| `log_trig` | log of serum triglycerides |
| `glucose` | blood sugar level |

★ Look at `X.describe()` and `y.describe()`.  How many patients and predictors are there?  Which predictor is not numerical?

In [3]:
print(X.describe())
print(y.describe())


              age         bmi          bp        chol         ldl         hdl  \
count  442.000000  442.000000  442.000000  442.000000  442.000000  442.000000   
mean    48.518100   26.375792   94.647014  189.140271  115.439140   49.788462   
std     13.109028    4.418122   13.831283   34.608052   30.413081   12.934202   
min     19.000000   18.000000   62.000000   97.000000   41.600000   22.000000   
25%     38.250000   23.200000   84.000000  164.250000   96.050000   40.250000   
50%     50.000000   25.700000   93.000000  186.000000  113.000000   48.000000   
75%     59.000000   29.275000  105.000000  209.750000  134.500000   57.750000   
max     79.000000   42.200000  133.000000  301.000000  242.400000   99.000000   

         chol_hdl    log_trig     glucose  
count  442.000000  442.000000  442.000000  
mean     4.070249    4.641411   91.260181  
std      1.290450    0.522391   11.496335  
min      2.000000    3.258100   58.000000  
25%      3.000000    4.276700   83.250000  
50%   

### Keeping a test set aside

When you use cross validation multiple times to *choose* between models, the cross validation scores of the chosen model become too optimistic.  For an honest final evaluation you need a separate test set that is used only once, at the very end.

★ Split `X` and `y` into a training set (75%) and a test set (25%), with `train_test_split(X, y, test_size=0.25, random_state=1)`.  Use the names `X_train`, `X_test`, `y_train`, `y_test`.  We will not touch `X_test` and `y_test` until section 4.

In [ ]:
from sklearn.model_selection import train_test_split



assert len(X_train) == 331 and len(X_test) == 111

## 1. A linear baseline model

### Different preprocessing for different columns

The numerical columns should be scaled, but the categorical column `sex` needs one-hot encoding.  A `ColumnTransformer` applies different preprocessing steps to different columns, and puts the results side by side:

```
from sklearn.compose import make_column_transformer
preprocessor = make_column_transformer(
    (StandardScaler(), numerical_columns),
    (OneHotEncoder(drop="if_binary"), ["sex"]),
    verbose_feature_names_out=False)
```

(`drop="if_binary"` makes a single 0/1 column for a variable with two categories, as on the one-hot encoding slide.  `verbose_feature_names_out=False` keeps the column names short.)

★ Make the list `numerical_columns` of all columns except `sex`, and build `preprocessor` as above.  Apply it to the training data with `preprocessor.fit_transform(X_train)`, and check the shape of the result.  Print the names of the new columns with `preprocessor.get_feature_names_out()`.

In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder



assert preprocessor.fit_transform(X_train).shape == (331, 10)

### Cross validation: training and test scores

★ Make a pipeline `linear_model` of `preprocessor` and a `LinearRegression`.  Evaluate it with 10-fold cross validation on the training data: `cv_results = cross_validate(linear_model, X_train, y_train, cv=10, return_train_score=True)`.  Print the mean of `cv_results["train_score"]` and of `cv_results["test_score"]` (both are R² values).  Is there much overfitting?

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_validate



assert 0.40 < cv_results["test_score"].mean() < 0.55

### The coefficients

★ Fit `linear_model` on the whole training set, and show its coefficients as a Series, with the names from `linear_model[0].get_feature_names_out()` as index.  Because the predictors are scaled, the coefficients are comparable: each is the change in progression for one standard deviation change in that predictor.
* Which predictors have the largest effect?
* Look at `chol` and `ldl`: their coefficients are large, and have opposite signs.  Compute the correlation between these two columns, with `X_train[["chol", "ldl"]].corr()`.  Can you explain the coefficients?

In [ ]:
assert len(linear_coefs) == 10

## 2. Nonlinear features

A linear model can describe nonlinear relationships if we add nonlinear features, such as squares (`bmi²`) and products (`bmi × bp`, an *interaction*).  `PolynomialFeatures(degree=2)` adds all squares and all pairwise products of its input columns.

★ Run the cell to see what `PolynomialFeatures` does with two small columns `a` and `b`.  Explain each output column.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

small = pd.DataFrame({"a": [1, 2, 3], "b": [10, 20, 30]})
poly = PolynomialFeatures(degree=2, include_bias=False)
pd.DataFrame(poly.fit_transform(small), columns=poly.get_feature_names_out())

We only want polynomial features of the numerical columns, not of the 0/1 column for `sex`.  As the slides say, we can do this with a `ColumnTransformer`: for the numerical columns, we use a small pipeline that first makes polynomial features and then scales them.

★ Build `poly_preprocessor`, like `preprocessor` above, but with `make_pipeline(PolynomialFeatures(degree=2, include_bias=False), StandardScaler())` instead of `StandardScaler()` for the numerical columns.  How many features does it produce for the training data?  Can you explain this number?  (Hint: 9 numerical columns.)

In [ ]:
assert n_features == 55

We will build many models of this form, with different degrees and different regression models.  The function below does this for us.

★ Run the cell, and read the function.  `make_poly_model(2, LinearRegression())` gives the same model as `make_pipeline(poly_preprocessor, LinearRegression())`.

In [ ]:
def make_poly_model(degree, regressor):
    poly_preprocessor = make_column_transformer(
        (make_pipeline(PolynomialFeatures(degree=degree, include_bias=False), StandardScaler()), numerical_columns),
        (OneHotEncoder(drop="if_binary"), ["sex"]),
    verbose_feature_names_out=False)
    return make_pipeline(poly_preprocessor, regressor)

★ For `degree` = 1, 2 and 3, make a model with `make_poly_model(degree, LinearRegression())`, run 10-fold cross validation as above, and print the degree, the mean train score and the mean test score.  (Use a `for` loop.)
* What happens to the train score as the degree increases?  And to the test score?
* For degree 3 the test score is negative.  What does a negative R² mean?
* How many features does the degree-3 model have, and how many patients are in each training fold?

In [ ]:
assert degree_scores[3] < 0 < degree_scores[1]

## 3. Regularization

Regularization adds a penalty for large coefficients to the loss function.  The strength of the penalty is a hyperparameter called `alpha` in scikit-learn: `alpha=0` means no penalty (ordinary linear regression); a large `alpha` forces the coefficients towards 0.

* **Ridge** regression (`sklearn.linear_model.Ridge`) uses an L2 penalty: it shrinks all coefficients.
* **Lasso** regression (`sklearn.linear_model.Lasso`) uses an L1 penalty: it sets some coefficients exactly to 0.

### Ridge

★ Use the degree-2 model, but with `Ridge(alpha=alpha)` instead of `LinearRegression()`, for `alpha` = 0.01, 1, 10, 100 and 1000.  Print alpha, the mean train score and the mean test score from 10-fold cross validation.
* Compare with the degree-2 linear regression from section 2.
* Which alpha gives the best test score?  What happens when alpha is very large?

In [ ]:
from sklearn.linear_model import Ridge



assert ridge_scores[100] > ridge_scores[0.01]

★ Now look at how Ridge shrinks the coefficients.  Fit `make_poly_model(1, Ridge(alpha=100))` (degree 1, so only the 10 original features) on the training data, and put its coefficients next to `linear_coefs` in a DataFrame with columns `linear` and `ridge`.  What happened to the coefficients of `chol` and `ldl`?

In [ ]:
assert abs(coefs["ridge"]["chol"]) < abs(coefs["linear"]["chol"])

### Lasso

★ Fit `make_poly_model(1, Lasso(alpha=5))` on the training data, and show its coefficients next to the linear ones.  Which coefficients are exactly 0?  Which predictors does the Lasso keep, and do they make sense medically?  Try a few other values of alpha (e.g. 1 and 10): how does the number of selected predictors change?

In [ ]:
from sklearn.linear_model import Lasso



assert (coefs["lasso"] == 0).sum() >= 4

## 4. Hyperparameter search

So far we chose the degree and alpha by hand.  Scikit-learn can search for good values for us, but first we need the *names* of these hyperparameters inside our pipeline.

### Hyperparameter names

★ Make `model = make_poly_model(2, Ridge())`, and print the names of all its hyperparameters with `model.get_params().keys()`.  Find the names of the polynomial degree and of alpha.  How are the names built up?

### Validation curve

A validation curve shows the cross-validated train and test scores for a range of values of **one** hyperparameter.

★ Run the cell.  It makes a validation curve for `ridge__alpha` of the degree-2 model, for alpha from 0.01 to 10,000.  Where does the model overfit?  Where does it underfit?  Which alpha would you choose?

In [ ]:
from sklearn.model_selection import ValidationCurveDisplay

ValidationCurveDisplay.from_estimator(
    make_poly_model(2, Ridge()), X_train, y_train,
    param_name="ridge__alpha", param_range=np.logspace(-2, 4, 13), cv=10)
plt.xscale("log")
plt.show()

### Grid search

To choose two hyperparameters at the same time, `GridSearchCV` tries all combinations, and evaluates each with cross validation.  After `fit`, it refits the best combination on all the training data, so the grid search object can be used as a model.

★ Make a `GridSearchCV` for `make_poly_model(2, Ridge())`, with `cv=10` and this parameter grid:

```
param_grid = {"columntransformer__pipeline__polynomialfeatures__degree": [1, 2, 3],
              "ridge__alpha": np.logspace(-2, 4, 13)}
```

Fit it on the training data.  Print `grid.best_params_` and `grid.best_score_`.  How many models were fitted in total?

In [ ]:
from sklearn.model_selection import GridSearchCV



assert grid.best_params_["columntransformer__pipeline__polynomialfeatures__degree"] in [1, 2]

★ The full results are in `grid.cv_results_`.  Turn them into a DataFrame, and show the columns `param_columntransformer__pipeline__polynomialfeatures__degree`, `param_ridge__alpha`, `mean_test_score` and `rank_test_score` for the 5 best combinations (use `sort_values("rank_test_score")`).  Are the best few combinations very different in score?

### The final, honest evaluation

★ Now, and only now, use the test set.  Compute `grid.score(X_test, y_test)`.  Compare it with `grid.best_score_`.  Why would you expect the test score to be a bit lower?

★ For comparison, also fit the simple `linear_model` from section 1 on the training data and compute its score on the test set.  Was all the extra effort worth it?  What does this tell you about the data?

In [ ]:
assert test_score < grid.best_score_

---
## ★★ Challenges

### Lasso with nonlinear features
Use `GridSearchCV` to find the best `alpha` for `make_poly_model(2, Lasso(max_iter=10_000))` (the parameter is now called `lasso__alpha`; try `np.logspace(-2, 1, 10)`).  Which of the 55 features does the best model keep?  (Use `grid_lasso.best_estimator_[0].get_feature_names_out()` and the coefficients of `grid_lasso.best_estimator_[-1]`.)  Are there squares or interactions among them?

### Randomized search
Instead of a grid, `RandomizedSearchCV` tries `n_iter` random combinations.  For alpha you then give a *distribution* instead of a list: `scipy.stats.loguniform(1e-2, 1e4)`.  Run a randomized search with `n_iter=20` over the degree (1, 2 or 3) and alpha.  Do you find similar values as the grid search?  How many models did it fit, compared with the grid search?

### Nested cross validation
The slides mention *nested* cross validation as an alternative to a separate test set: the complete hyperparameter search is repeated inside each fold of an outer cross validation.  Run `cross_validate(grid, X_train, y_train, cv=5)` (this fits the grid search 5 times, so it takes a little while).  Compare the mean outer test score with `grid.best_score_`.